# Time-resolved characterization with `edges`

This notebook shows `TimexLCA.edges_lcia()`, which characterizes a time-explicit inventory with
the [`edges`](https://edges.readthedocs.io) package: every exchange is characterized with the
characterization factor (CF) of the **year in which it occurs**, instead of a single static CF.

The example database has one foreground process, "heat production", that emits 10 kg CO2. That
emission is not instantaneous: a temporal distribution spreads it 40 % / 60 % across two
consecutive calendar years, even though the process itself runs in a single year. This is exactly
the situation `edges_lcia()` is built for:

- **Biosphere flows** are characterized at the **date of the emission**, taken from `bw_timex`'s
  dynamic inventory. A temporal distribution on a biosphere exchange is respected, which the
  static, expanded biosphere matrix cannot represent (it has one undated cell per flow/process).
- **Technosphere flows** are characterized at the **vintage of the consuming process**, because
  `bw_timex` has no dynamic technosphere inventory. This is a documented limitation, not a knob
  (see the last section).

This is **orthogonal** to `TimexLCA.dynamic_lcia()`: `edges` varies the *characterization factor*
with the year of the exchange, while `dynamic_lcia()` varies the *impact* with the time elapsed
since the emission (e.g. radiative forcing decaying after a CH4 pulse). The two can be combined,
but this notebook only shows `edges_lcia()`.

`edges` is an optional extra: `pip install bw_timex[edges]` (Python 3.11/3.12 only, `edges`
requires `<3.13`).

## 1. Build a small example database

The data below is built directly in a scratch Brightway project so this notebook is
self-contained and does not require ecoinvent. It follows the same pattern as `bw_timex`'s own
test fixtures: one biosphere flow (CO2), one background process ("electricity production", dated
2020) and one foreground process ("heat production") that consumes some of that electricity and
emits CO2 with a temporal distribution.

In [1]:
import bw2data as bd
import numpy as np
import pandas as pd
from bw_temporalis import TemporalDistribution
from datetime import datetime

project_name = "timex_example_edges_characterization"
if project_name in bd.projects:
    bd.projects.delete_project(project_name)  # making sure to start from scratch
    bd.projects.purge_deleted_directories()

bd.projects.set_current(project_name)

bd.Database("bio").write(
    {
        ("bio", "CO2"): {
            "type": "emission",
            "name": "carbon dioxide",
            "categories": ("air",),
            "unit": "kilogram",
        },
    },
)

bd.Database("db_2020").write(
    {
        ("db_2020", "electricity"): {
            "name": "electricity production",
            "location": "CH",
            "reference product": "electricity",
            "unit": "kilowatt hour",
            "type": "process",
            "exchanges": [
                {"amount": 1, "type": "production", "input": ("db_2020", "electricity")},
                {"amount": 2, "type": "biosphere", "input": ("bio", "CO2")},
            ],
        },
    }
)

bd.Database("foreground").write(
    {
        ("foreground", "heat"): {
            "name": "heat production",
            "location": "CH",
            "reference product": "heat",
            "unit": "megajoule",
            "type": "process",
            "exchanges": [
                {"amount": 1, "type": "production", "input": ("foreground", "heat")},
                {
                    "amount": 10,
                    "type": "biosphere",
                    "input": ("bio", "CO2"),
                    # 10 kg CO2 split 40 % / 60 % across two consecutive calendar years.
                    "temporal_distribution": TemporalDistribution(
                        date=np.array([0, 366], dtype="timedelta64[D]"),
                        amount=np.array([0.4, 0.6]),
                    ),
                },
                {"amount": 3, "type": "technosphere", "input": ("db_2020", "electricity")},
            ],
        },
    }
)

bd.Method(("GWP", "example")).write([(("bio", "CO2"), 1)])

11:56:36+0200 [warning  ] Removing project from project timex_example_edges_characterization list, but not deleting data; if you switch to this project again you will have the same data again. To delete data permanently, pass `(..., delete_dir=True)`.


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 8144.28it/s]

11:56:36+0200 [info     ] Vacuuming database            


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 22310.13it/s]

11:56:36+0200 [info     ] Vacuuming database            


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 23831.27it/s]

11:56:36+0200 [info     ] Vacuuming database            


## 2. Build the `TimexLCA` and calculate the time-explicit inventory

`edges_lcia()` needs both the expanded technosphere matrix (the default) and the dynamic
biosphere inventory (also the default), since it reads emission dates from
`TimexLCA.dynamic_inventory`.

In [2]:
from bw_timex import TimexLCA

node = bd.get_node(database="foreground", code="heat")

tlca = TimexLCA(
    demand={node: 1},
    method=("GWP", "example"),
    database_dates={
        "db_2020": datetime.strptime("2020", "%Y"),
        "foreground": "dynamic",
    },
)
tlca.build_timeline(starting_datetime=datetime(2024, 1, 1))
tlca.lci()
tlca.static_lcia()

print(f"static_score: {tlca.static_score}")

2026-09-13 11:56:36.925 | INFO     | bw_timex.timex_lca:__init__:137 - Initializing TimexLCA object...


2026-09-13 11:56:36.926 | INFO     | bw_timex.timex_lca:__init__:154 - Calculating base LCA...


2026-09-13 11:56:36.933 | INFO     | bw_timex.timex_lca:__init__:171 - Collecting node infos...


2026-09-13 11:56:36.934 | INFO     | bw_timex.timex_lca:build_timeline:343 - No edge filter function provided. Skipping all edges in background databases.


2026-09-13 11:56:36.935 | INFO     | bw_timex.timex_lca:build_timeline:364 - Creating activity time mapping...


...

2026-09-13 11:56:36.938 | INFO     | bw_timex.timeline_builder:build_timeline:186 - Building timeline...


2026-09-13 11:56:36.950 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:630 - Reference date 2024-01-01 00:00:00 is higher than all provided dates. Data will be taken from the closest lower year.


2026-09-13 11:56:36.956 | INFO     | bw_timex.timex_lca:lci:514 - Expanding matrices...


2026-09-13 11:56:36.959 | INFO     | bw_timex.timex_lca:lci:533 - Calculating dynamic inventory...


Starting graph traversal
Calculation count: 1
static_score: 16.0


The static score characterizes the *entire* 10 kg + 3 kWh * 2 kg/kWh background CO2 = 16 kg CO2
at a single CF, ignoring when each kg was actually emitted.

## 3. Define a year-dependent CF and run `edges_lcia()`

`edges` methods are plain dictionaries (or JSON files) of characterization factors. A CF can be a
symbolic expression (`value_expression`) evaluated per scenario year, with the actual numbers
supplied via `parameters` in the shape `{scenario: {parameter: {year: value}}}`. Here CO2 is worth
1.0 in 2024 and 2.0 in 2025, so the CF used depends on *when* the exchange occurs.

In [3]:
edges_method = {
    "name": "example year-dependent CF",
    "version": "1.0",
    "unit": "kg CO2-eq",
    "interpolation": {
        "axis": "scenario_idx",
        "axis_type": "year",
        "method": "linear",
        "extrapolation": "nearest",
    },
    "exchanges": [
        {
            "supplier": {"name": "carbon dioxide", "categories": ["air"], "matrix": "biosphere"},
            "consumer": {"matrix": "technosphere"},
            "value": 1.0,
            "value_expression": "cf_co2",
        }
    ],
}

parameters = {"example": {"cf_co2": {"2024": 1.0, "2025": 2.0}}}

table = tlca.edges_lcia(
    method=edges_method,
    parameters=parameters,
    scenario="example",
    regionalized=False,
)
table

,supplier,supplier categories,consumer,consumer location,activity,direction,date,year,amount,CF,impact
0,carbon dioxide,"(air,)",electricity production,CH,357464577675784193,biosphere-technosphere,2024-01-01,2024,6.0,1.0,6.0
1,carbon dioxide,"(air,)",heat production,CH,357464577675784194,biosphere-technosphere,2024-01-01,2024,4.0,1.0,4.0
2,carbon dioxide,"(air,)",heat production,CH,357464577675784194,biosphere-technosphere,2025-01-01,2025,6.0,2.0,12.0


## 4. Compare to the static score

Grouping the result by `year` shows the effect directly: the 4 kg emitted in 2024 are worth 4 * 1.0
= 4, the 6 kg emitted in 2025 are worth 6 * 2.0 = 12, for a foreground total of 16. The background
electricity's 6 kg CO2 has no temporal distribution of its own, so its emission date is inherited
from the consuming process (2024) and it is characterized at CF 1.0, adding 6. The total,
`edges_score`, is 22 — higher than the static score of 16, because half the foreground emission
happens in the year with the doubled CF.

In [4]:
print(table.groupby("year")["impact"].sum())
print()
print(f"static_score: {tlca.static_score}")
print(f"edges_score:  {tlca.edges_score}")

year
2024    10.0
2025    12.0
Name: impact, dtype: float64

static_score: 16.0
edges_score:  22.0


## 5. Limitations

- **Technosphere CFs use the consuming process's vintage, not an emission date.** `bw_timex`
  builds a dynamic inventory only for biosphere flows; there is no dynamic technosphere
  inventory. A technosphere characterization factor (e.g. a resource or land-use CF attached to a
  product exchange rather than an elementary flow) is therefore always evaluated at the year of
  the process that consumes it, even if that exchange itself has a temporal distribution.
- **Temporal markets are characterized once, not twice.** `bw_timex` inserts a "temporal market"
  node between a consuming process and its background supplier to blend supply across database
  vintages. That market column carries the supplying commodity's own identity (same name,
  reference product and location), so a technosphere CF pattern that matches the real supplier
  would also match the edge into the market — and again on the market's own downstream edge to
  the consumer, double-counting one physical flow. `edges_lcia()` drops the edges *into* a
  temporal market and characterizes only the market's edge to the consumer, at the consuming
  process's vintage. This changes the technosphere score compared to a naive per-edge
  characterization, so keep it in mind when comparing numbers against a different tool.
- **CF uncertainty (`use_distributions`) is not supported.** Per-year characterization evaluates a
  single characterization matrix per year; a distribution would need a third dimension that this
  code path does not build. Passing `use_distributions=True` to the underlying adapter raises
  `NotImplementedError`.
- **The timeline route is not supported.** `edges_lcia()` requires the expanded, time-explicit
  matrices built by `TimexLCA.lci()` (the default). Calling `TimexLCA.lci(expand_technosphere=False)`
  first and then `edges_lcia()` raises `NotImplementedError`, since that route has no expanded
  technosphere matrix to characterize.